In [1]:
import json 
from pathlib import Path
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from metrics import (
    mean_average_precision_score,
    F1_score_from_sim,
    IoU_from_sim,
    IoU_q
)

%load_ext autoreload
%autoreload 2

In [2]:
output_folder = Path("/media/eceo_scratch_haas001/results/CLIP4Clip/test/")
gt_json = Path("../zero_shot/CLIP4Clip/dataset/test/ground_truth.json")
with open(gt_json, "r") as f:
    gt_queries_videos = json.load(f)
queries = gt_queries_videos.keys()
sim_matrix_df = pd.read_csv(output_folder / "similarity_matrix.csv").set_index("video_id")
sim_matrix_df

,An animal engaged in any activity other than foraging.,An animal that is neither a red deer nor a roe deer.,An animal running.,An animal bathing.,A roe deer grazing.,An animal browsing.,An adult roe deer sniffing.,A juvenile red deer scratching its body.,A chamois trotting.,An adult male roe deer jumping.,...,An animal running while reacting to a camera.,An animal looking at a camera.,An animal reacting to a camera and then foraging.,"An animal foraging, then reacting to a camera, and then returning to foraging.",An animal reacting to a camera and then running away.,"An animal foraging, then reacting to a camera, and then running away.",An animal reacting to a camera in rainy or overcast weather.,An animal reacting to a camera in clear or sunny weather.,A single adult red deer foraging only.,An empty video.
video_id,,,,,,,,,,,,,,,,,,,,,
S1_C1_E100_V0251,22.025024,23.058090,23.182589,19.843180,22.673430,20.422297,22.582565,21.165018,23.312788,22.198380,...,22.866163,19.838554,25.505990,26.224625,23.038357,25.728630,26.414658,23.696938,25.384617,24.381996
S1_C1_E103_V0253,25.327059,25.812813,24.681633,19.453003,25.737038,23.393959,24.397135,24.020302,22.956180,24.664598,...,26.388105,23.744965,28.624557,28.623095,27.176434,29.404380,24.152350,21.749860,26.057178,22.562260
S1_C1_E103_V0254,27.366703,27.258469,25.552150,20.729197,27.250528,24.342224,26.422905,26.267862,25.795517,25.703081,...,26.886086,24.648014,29.865953,29.831013,27.782090,30.702028,24.998377,22.821041,27.432146,20.780462
S1_C1_E103_V0255,26.163033,26.442957,24.768353,20.247803,26.607132,24.112444,25.912556,25.335983,24.365292,25.056978,...,26.395887,24.453623,29.000748,28.851583,27.379921,29.891031,24.234856,21.930916,26.565964,21.234852
S1_C1_E103_V0256,25.339710,25.402850,24.501010,19.609590,25.393260,23.367579,24.657450,23.956980,22.942724,24.738043,...,26.289034,23.753078,28.601917,28.534672,27.102724,29.495611,23.785587,21.538057,25.635607,22.011032
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
S3_C5_F719_V0154,25.470102,26.937819,24.194810,21.579111,28.781870,24.375721,28.479935,26.792372,23.607350,25.945854,...,26.644240,25.773058,29.412035,28.974236,27.417547,29.341835,26.338520,24.864092,27.032345,21.845661
S3_C5_F719_V0155,25.468456,26.781973,24.589520,21.392715,28.630522,24.555481,28.577590,26.747175,24.265448,26.501148,...,26.879490,25.703505,29.425930,28.922356,27.489489,29.292034,25.960180,24.640420,26.998362,22.137127
S3_C5_F719_V0156,25.220602,26.415209,24.586687,21.490332,28.541178,24.518880,28.391330,26.153364,24.014305,26.312240,...,26.471355,25.480057,29.034618,28.603899,27.085728,28.868706,25.779320,24.389738,26.638830,22.406485


# mAP

In [3]:
mAP, APs = mean_average_precision_score(gt_queries_videos, sim_matrix_df)
print(f"mAP queries: {mAP: .3f}")
sorted(APs.items(), key=lambda x:x[1], reverse=True)

mAP queries:  0.089


[('An animal being vigilant while the weather is rainy or overcast.',
  0.5815419249864335),
 ('An animal engaged in any activity other than foraging.',
  0.5642802407400905),
 ('Rainy weather.', 0.3875653699117429),
 ('A single adult red deer foraging only.', 0.38682948634615816),
 ('An adult red deer standing with its head up while participating in courtship.',
  0.3103757372141977),
 ('An adult red deer vocalizing while participating in courtship.',
  0.2996077322123413),
 ('An animal participating in courtship.', 0.2582176148545082),
 ('A video of three or more red deer.', 0.24425774164699032),
 ('A red deer foraging in sunny weather.', 0.22771518176472808),
 ('A red deer foraging in rainy weather.', 0.22329565563352702),
 ('A juvenile red deer playing.', 0.222275641025641),
 ('An adult male red deer being vigilant after vocalizing.',
  0.21775261485911168),
 ('Two or more animals reacting to a camera.', 0.2),
 ('A video of two or more animals.', 0.19350317278504692),
 ('An animal 

# mIoU and F1-score (require threshold)

Find the optimal threshold per query on train set

In [6]:
gt_json_train = Path("../zero_shot/CLIP4Clip/dataset/train/ground_truth.json")
output_folder_train = Path("/media/eceo_scratch_haas001/results/CLIP4Clip/train")
with open(gt_json_train, "r") as f:
    gt_queries_videos_train = json.load(f)
queries = gt_queries_videos_train.keys()
sim_matrix_train_df = pd.read_csv(output_folder_train / "similarity_matrix.csv").set_index("video_id")
sim_matrix_train_df

,An animal engaged in any activity other than foraging.,An animal that is neither a red deer nor a roe deer.,An animal running.,An animal bathing.,A roe deer grazing.,An animal browsing.,An adult roe deer sniffing.,A juvenile red deer scratching its body.,A chamois trotting.,An adult male roe deer jumping.,...,An animal looking at a camera.,A wolf reacting to a camera.,An animal reacting to a camera and then foraging.,"An animal foraging, then reacting to a camera, and then returning to foraging.",An animal reacting to a camera and then running away.,"An animal foraging, then reacting to a camera, and then running away.",An animal reacting to a camera in rainy or overcast weather.,An animal reacting to a camera in clear or sunny weather.,A single adult red deer foraging only.,An empty video.
video_id,,,,,,,,,,,,,,,,,,,,,
S1_C1_E104_V0262,25.047344,25.285330,24.552656,19.309107,25.557714,23.111443,24.132568,23.560472,23.293472,24.497710,...,23.193817,26.580597,28.561604,28.724882,26.789042,29.312930,24.176136,21.585636,25.889946,22.138681
S1_C1_E104_V0263,25.129768,25.339550,24.524174,19.191492,25.660034,23.132143,24.256533,23.835802,23.466125,24.599577,...,23.208872,26.523512,28.671432,28.804660,26.872340,29.384876,23.854362,21.573840,26.074451,22.180128
S1_C1_E104_V0264,25.215635,25.479912,24.454012,19.105349,25.930750,23.100508,24.516160,24.137667,23.504972,24.822218,...,23.219446,26.515522,28.687931,28.771246,26.810583,29.366306,23.872883,21.607199,26.286346,22.095740
S1_C1_E104_V0265,24.801502,25.055367,24.013224,19.131226,25.326508,22.826614,23.900690,23.351688,22.770237,24.287825,...,22.743216,26.349660,28.538345,28.910583,26.625925,29.485788,23.873444,21.284311,25.750320,22.330738
S1_C1_E104_V0266,24.824814,25.172010,24.430773,19.113312,25.508318,22.973051,23.982685,23.508600,22.924400,24.311130,...,22.993132,26.311255,28.442360,28.654593,26.644493,29.164284,23.868908,21.408472,25.840040,22.268000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
S3_C5_F930_V0504,25.411633,23.717402,26.792934,21.728666,25.432741,23.784174,24.181986,24.825504,26.065847,24.998274,...,23.933980,26.835802,27.266258,27.431879,26.529785,27.983269,21.944809,26.905596,27.454037,26.286762
S3_C5_F934_V0508,25.585928,23.728724,25.865635,20.917130,23.433332,23.601980,23.073425,24.126307,28.579414,23.537483,...,24.173534,24.899780,26.104073,25.758759,24.384851,25.410519,21.625284,25.993470,25.416970,22.962930
S3_C5_F953_V0782,25.353405,23.930891,26.264364,20.716549,24.314304,23.411760,23.492044,24.640018,28.975788,23.809994,...,23.889204,24.640005,25.862047,25.570848,23.809603,24.957440,21.234713,25.961480,26.143545,23.008465


In [7]:
thrs = np.arange(sim_matrix_train_df.values.min(), sim_matrix_train_df.values.max(), step=0.5)
best_t = dict.fromkeys(gt_queries_videos_train.keys())
for q in gt_queries_videos_train.keys():
    best_IoU = 0
    if q not in sim_matrix_train_df.columns:
        print(f"Query {q} has not been processed")
        continue
    gt_videos_q = list(gt_queries_videos_train[q]["videos"])
    for t in thrs:
        ass_videos_q = list(sim_matrix_train_df.index[sim_matrix_train_df[q] > t])
        iou = IoU_q(gt_videos_q, ass_videos_q)
        if not np.isnan(iou) and iou > best_IoU:
            best_IoU = iou
            best_t[q] = t
    
    if best_t[q] is None:
        print(f"No best threshold found for {q}, using median similarity score as threshold")
        best_t[q] = np.median(sim_matrix_train_df.values)

In [ ]:
mF1, F1_queries = F1_score_from_sim(gt_queries_videos=gt_queries_videos, sim_matrix_df=sim_matrix_df, best_thresholds=best_t)
mIoU, IoU_queries = IoU_from_sim(gt_queries_videos=gt_queries_videos, sim_matrix_df=sim_matrix_df, best_thresholds=best_t)

In [19]:
print(f"F1-score (macro-avg.): {mF1:.2f}")
print(f"mean IoU: {mIoU:.2f}")

F1-score (macro-avg.): 0.09
mean IoU: 0.05


In [33]:
results_df = pd.concat([
    pd.DataFrame.from_dict(IoU_queries, orient="index", columns=["mIoU"]),
    pd.DataFrame.from_dict(F1_queries, orient="index", columns=["F1-score"])
], axis=1)

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)
results_df.sort_values("mIoU", ascending=False)

,mIoU,F1-score
A single adult red deer foraging only.,0.476129,0.645105
An animal being vigilant while the weather is rainy or overcast.,0.406250,0.577778
An animal engaged in any activity other than foraging.,0.404092,0.575592
An adult red deer standing with its head up while participating in courtship.,0.250000,0.400000
An animal participating in courtship.,0.228070,0.371429
An adult red deer vocalizing while participating in courtship.,0.225564,0.368098
A video of an animal drinking.,0.187500,0.315789
An adult male red deer being vigilant after vocalizing.,0.171875,0.293333
An animal being vigilant while the weather is clear or sunny.,0.150968,0.262332
Rainy weather.,0.139355,0.244621
